copy pasted this to test if the model works

In [1]:
from pathlib import Path
from pydantic_ai.messages import BinaryContent
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
import os

model = OpenAIChatModel(
    model_name="nvidia/nemotron-3.5-super-vl-120b-a12b",
    provider=OpenAIProvider(
        base_url="https://integrate.api.nvidia.com/v1", api_key=os.environ["NVIDIA_API"]
    ),
)

image_joker = Agent(
    model=model,
    instructions="You are a person that can analyze an image, explain the contents and make a joke about it, max 3 sentences.",
)

image_data = Path("bella.png").read_bytes()
prompt = "tell me about this dude"

result = await image_joker.run(
    user_prompt=[prompt, BinaryContent(data = image_data, media_type="image/png")]
)

result.output

'This is a grey rabbit fast asleep while wearing a tiny sailor hat and a blue and yellow ribbon labeled "Festive." The bunny looks like he missed the memo that formal attire requires you to actually keep your eyes open. Honestly, he looks more committed to napping than to whatever nautical adventure that ribbon promises.'

making classes so it can read out recipies based on pictures of ingredients

In [24]:
from pydantic import BaseModel, Field

class Ingredient(BaseModel):
    name: str
    description: str = Field(description="the apparance of how the ingredient generally looks")

class Recipe(BaseModel):
    name: str
    instructions: str = Field(description="simple and clear intstructions on how to make the dish tied to the recipe")
    ingredient_list: list[Ingredient] = Field(description="list of the ingredients in the dish and the amounts needed to cook the recipe, use the metric system")
    servings: int = Field(description="the amount of servings of the dish")

class Fridge(BaseModel):
    available_ingredients: list[Ingredient] = Field(description="list of visible ingredients")
    description: str = Field(description="describe the inside of the fridge")
    recipes: list[Recipe] = Field(description="list of all available recipes based on visible ingredients")


image_chef = Agent(
    model=model,
    instructions="You are a chef that can analyze an image of an open fridge, come up with a recipe that involves the ingredients that are visible in the image provided. "
                "you are allowed to provide recipes that use ingredients currently missing from the image, but you need to state that the ingredient is missing and the user needs to buy more",
)

image_data = Path("fridge.png").read_bytes()
prompt = "give me a few recipes I can make with the things in my fridge"

result = await image_chef.run(
    user_prompt=[prompt, BinaryContent(data = image_data, media_type="image/png")],
    output_type=Fridge
)

content = result.output

In [25]:
content

Fridge(available_ingredients=[Ingredient(name='Eggs', description='Carton of brown eggs on the middle shelf'), Ingredient(name='Branston Small Chunk Pickle', description='Jar of brown pickle relish on the middle shelf'), Ingredient(name='Pickled peppers (Feferoni)', description='Jar of pickled peppers on the middle shelf'), Ingredient(name='Yellow peppers in bowl', description='Sliced yellow peppers or pickled peppers in a glass bowl covered with plastic wrap'), Ingredient(name='Red jarred sauce/condiment', description='Jar with red sauce or jam on the right side of middle shelf'), Ingredient(name='Milk', description='Cartons of ICA Mjölk milk on the bottom shelf'), Ingredient(name='Whipping cream', description='Carton of Vispgrädde whipping cream on the bottom shelf'), Ingredient(name='Cream cheese', description='Tub of cream cheese spread with a blue spoon on the middle shelf'), Ingredient(name='Yogurt', description='Containers of yogurt on the bottom shelf'), Ingredient(name='Caulif

In [27]:
content.recipes[0]

Recipe(name='Creamy Cauliflower, Pepper & Egg Frittata', instructions='Preheat oven to 180°C. Chop the cauliflower into small florets and dice the yellow bell pepper. Sauté both in olive oil over medium heat until softened, about 8-10 minutes. In a bowl, whisk 6 eggs with 100ml milk and 2 tablespoons cream cheese until smooth. Season with salt and pepper. Pour the egg mixture over the vegetables in an oven-safe skillet. Cook on the stovetop for 2-3 minutes until the edges set, then transfer to the oven and bake for 15-20 minutes until golden and set in the center. Let cool slightly before slicing. Serve warm or at room temperature.', ingredient_list=[Ingredient(name='Eggs', description='6 large eggs'), Ingredient(name='Milk', description='100ml'), Ingredient(name='Cream cheese', description='2 tablespoons'), Ingredient(name='Cauliflower', description='300g, cut into small florets'), Ingredient(name='Yellow bell pepper', description='1 medium, diced'), Ingredient(name='Olive oil', descr

In [32]:
content.recipes[0].ingredient_list

[Ingredient(name='Eggs', description='6 large eggs'),
 Ingredient(name='Milk', description='100ml'),
 Ingredient(name='Cream cheese', description='2 tablespoons'),
 Ingredient(name='Cauliflower', description='300g, cut into small florets'),
 Ingredient(name='Yellow bell pepper', description='1 medium, diced'),
 Ingredient(name='Olive oil', description='1 tablespoon'),
 Ingredient(name='Branston Small Chunk Pickle', description='1 teaspoon (optional, stirred into eggs for tangy depth)')]

In [33]:
content.recipes[0].instructions

'Preheat oven to 180°C. Chop the cauliflower into small florets and dice the yellow bell pepper. Sauté both in olive oil over medium heat until softened, about 8-10 minutes. In a bowl, whisk 6 eggs with 100ml milk and 2 tablespoons cream cheese until smooth. Season with salt and pepper. Pour the egg mixture over the vegetables in an oven-safe skillet. Cook on the stovetop for 2-3 minutes until the edges set, then transfer to the oven and bake for 15-20 minutes until golden and set in the center. Let cool slightly before slicing. Serve warm or at room temperature.'

In [29]:
content.available_ingredients

[Ingredient(name='Eggs', description='Carton of brown eggs on the middle shelf'),
 Ingredient(name='Branston Small Chunk Pickle', description='Jar of brown pickle relish on the middle shelf'),
 Ingredient(name='Pickled peppers (Feferoni)', description='Jar of pickled peppers on the middle shelf'),
 Ingredient(name='Yellow peppers in bowl', description='Sliced yellow peppers or pickled peppers in a glass bowl covered with plastic wrap'),
 Ingredient(name='Red jarred sauce/condiment', description='Jar with red sauce or jam on the right side of middle shelf'),
 Ingredient(name='Milk', description='Cartons of ICA Mjölk milk on the bottom shelf'),
 Ingredient(name='Whipping cream', description='Carton of Vispgrädde whipping cream on the bottom shelf'),
 Ingredient(name='Cream cheese', description='Tub of cream cheese spread with a blue spoon on the middle shelf'),
 Ingredient(name='Yogurt', description='Containers of yogurt on the bottom shelf'),
 Ingredient(name='Cauliflower', description=

In [20]:
content.description

'A well-stocked refrigerator with multiple shelves containing dairy products, condiments, fresh vegetables in the crisper drawer, eggs, bread, and various jars and cartons on the door shelves.'